In [1]:
# ============================================
# 1. IMPORTS
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
# 2. Charger les données
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

# 3. Premier regard
print("Train shape :", train.shape)
print("Test shape :", test.shape)
print("\nColonnes train :", train.columns.tolist())
print("Colonnes test :", test.columns.tolist())

Train shape : (1290081, 11)
Test shape : (430100, 10)

Colonnes train : ['id', 'period', 'operation', 'amount', 'origin_account', 'origin_balance_before', 'origin_balance_after', 'destination_account', 'destination_balance_before', 'destination_balance_after', 'fraud_flag']
Colonnes test : ['id', 'period', 'operation', 'amount', 'origin_account', 'origin_balance_before', 'origin_balance_after', 'destination_account', 'destination_balance_before', 'destination_balance_after']


In [3]:
print("=== FEATURE ENGINEERING ===")

train_clean = train.copy()
test_clean = test.copy()

# 1. Target Encoding OOF (sécurisé)
def target_encoding_oof(train, test, cat_col, smoothing=150):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_te = pd.Series(0.0, index=train.index)
    
    for tr_idx, val_idx in skf.split(train, train['fraud_flag']):
        tr = train.iloc[tr_idx]
        stats = tr.groupby(cat_col)['fraud_flag'].mean()
        count = tr.groupby(cat_col).size()
        gm = tr['fraud_flag'].mean()
        smoothed = (stats * count + gm * smoothing) / (count + smoothing)
        train_te.iloc[val_idx] = train.iloc[val_idx][cat_col].map(smoothed).fillna(gm)
    
    stats_full = train.groupby(cat_col)['fraud_flag'].mean()
    count_full = train.groupby(cat_col).size()
    gm = train['fraud_flag'].mean()
    smoothed_full = (stats_full * count_full + gm * smoothing) / (count_full + smoothing)
    
    train[f'{cat_col}_te'] = train_te
    test[f'{cat_col}_te'] = test[cat_col].map(smoothed_full).fillna(gm)
    return train, test

train_clean, test_clean = target_encoding_oof(train_clean, test_clean, 'origin_account')
train_clean, test_clean = target_encoding_oof(train_clean, test_clean, 'destination_account')

# 2. Features simples (sans cible)
train_clean['amount_z_score'] = (train_clean['amount'] - train_clean['amount'].mean()) / train_clean['amount'].std()
test_clean['amount_z_score'] = (test_clean['amount'] - train_clean['amount'].mean()) / train_clean['amount'].std()

account_avg = train_clean.groupby('origin_account')['amount'].mean()
train_clean['amount_vs_account_avg'] = train_clean['amount'] / train_clean['origin_account'].map(account_avg)
test_clean['amount_vs_account_avg'] = test_clean['amount'] / test_clean['origin_account'].map(account_avg)
train_clean['amount_vs_account_avg'] = train_clean['amount_vs_account_avg'].fillna(1)
test_clean['amount_vs_account_avg'] = test_clean['amount_vs_account_avg'].fillna(1)

account_freq = train_clean.groupby('origin_account').size()
train_clean['origin_account_freq'] = train_clean['origin_account'].map(account_freq)
test_clean['origin_account_freq'] = test_clean['origin_account'].map(account_freq).fillna(1)

dest_freq = train_clean.groupby('destination_account').size()
train_clean['dest_account_freq'] = train_clean['destination_account'].map(dest_freq)
test_clean['dest_account_freq'] = test_clean['destination_account'].map(dest_freq).fillna(1)

# Balance to amount ratio avec log (beaucoup plus robuste)
train_clean['balance_to_amount_ratio'] = np.log1p(train_clean['origin_balance_before'] / (train_clean['amount'] + 1))
test_clean['balance_to_amount_ratio'] = np.log1p(test_clean['origin_balance_before'] / (test_clean['amount'] + 1))
# train_clean['balance_to_amount_ratio'] = (train_clean['origin_balance_before'] / (train_clean['amount'] + 1)).clip(0, 1000)
# test_clean['balance_to_amount_ratio'] = (test_clean['origin_balance_before'] / (test_clean['amount'] + 1)).clip(0, 1000)

train_clean['balance_change'] = train_clean['origin_balance_before'] - train_clean['origin_balance_after']
test_clean['balance_change'] = test_clean['origin_balance_before'] - test_clean['origin_balance_after']

train_clean['origin_dest_diff'] = (train_clean['origin_balance_before'] - train_clean['destination_balance_before']).abs()
test_clean['origin_dest_diff'] = (test_clean['origin_balance_before'] - test_clean['destination_balance_before']).abs()

# Taux de fraude par période
period_fraud_rate = train_clean.groupby('period')['fraud_flag'].mean()
train_clean['period_fraud_rate'] = train_clean['period'].map(period_fraud_rate)
test_clean['period_fraud_rate'] = test_clean['period'].map(period_fraud_rate).fillna(period_fraud_rate.mean())


features = [
    'amount', 'amount_z_score', 'amount_vs_account_avg',
    'origin_account_freq', 'origin_account_te', 
    'dest_account_freq', 'destination_account_te', 
    'balance_to_amount_ratio', 'balance_change', 'origin_dest_diff', 'period_fraud_rate'
]

print(f"Features : {len(features)}")

# ============================================
# 3. VALIDATION + ENTRAÎNEMENT
# ============================================
X = train_clean[features]
y_arr = train_clean['fraud_flag'].values
X_test = test_clean[features]

neg_count = (y_arr == 0).sum()
pos_count = (y_arr == 1).sum()
scale_pos_weight = neg_count / pos_count

xgb_params = {
    'n_estimators': 5000,        # réduit pour limiter l’overfitting
    'learning_rate': 0.015,
    'max_depth': 5,              # profondeur limitée (évite -1)
    'min_child_weight': 20,      # splits plus robustes
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_lambda': 15,
    'reg_alpha': 5,
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'logloss',
    'verbosity': 0,
    'early_stopping_rounds': 120,
    'random_state': 42,
}

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
scores_ap = []
scores_auc = []

print("\nEntraînement 5-fold XGBoost...")
for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y_arr)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y_arr[tr_idx],  y_arr[val_idx]

    model = XGBClassifier(**xgb_params)
    
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              verbose=False)

    val_pred = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_pred
    test_preds += model.predict_proba(X_test)[:, 1] / 5

    ap  = average_precision_score(y_val, val_pred)
    auc = roc_auc_score(y_val, val_pred)
    scores_ap.append(ap)
    scores_auc.append(auc)
    print(f"  Fold {fold+1}/5 — AP = {ap:.5f}  |  ROC-AUC = {auc:.5f}  |  best_iter = {model.best_iteration}")
    del model; gc.collect()

oof_ap  = average_precision_score(y_arr, oof_preds)
oof_auc = roc_auc_score(y_arr, oof_preds)

print(f"\n{'='*58}")
print(f"  OOF Average Precision (PR-AUC) : {oof_ap:.5f}   ← métrique officielle")
print(f"  OOF ROC-AUC                    : {oof_auc:.5f}")
print(f"  CV Mean AP  : {np.mean(scores_ap):.5f} ± {np.std(scores_ap):.5f}")
print(f"  CV Mean AUC : {np.mean(scores_auc):.5f} ± {np.std(scores_auc):.5f}")
print(f"{'='*58}")

=== FEATURE ENGINEERING V19 (corrigé) ===
Features : 12

Entraînement 5-fold XGBoost...
  Fold 1/5 — AP = 0.44700  |  ROC-AUC = 0.89567  |  best_iter = 4999
  Fold 2/5 — AP = 0.45184  |  ROC-AUC = 0.89791  |  best_iter = 4998
  Fold 3/5 — AP = 0.45253  |  ROC-AUC = 0.89716  |  best_iter = 4999
  Fold 4/5 — AP = 0.45523  |  ROC-AUC = 0.89842  |  best_iter = 4997
  Fold 5/5 — AP = 0.45698  |  ROC-AUC = 0.89855  |  best_iter = 4998

  OOF Average Precision (PR-AUC) : 0.45268   ← métrique officielle
  OOF ROC-AUC                    : 0.89754
  CV Mean AP  : 0.45272 ± 0.00341
  CV Mean AUC : 0.89754 ± 0.00106


In [4]:
# ============================================
# 8. SOUMISSION
# ============================================

submission = pd.DataFrame({
    'id': test_clean['id'],
    'target': test_preds  
})


submission.to_csv('submission_v15.csv', index=False) 

print(f"\n Soumission sauvegardée dans avec succee !")
print(f"   {len(submission)} transactions")
print(f"   Probabilité moyenne : {submission['target'].mean():.4f}")
print(f"   Probabilité min : {submission['target'].min():.4f}")
print(f"   Probabilité max : {submission['target'].max():.4f}")


 Soumission sauvegardée dans avec succee !
   430100 transactions
   Probabilité moyenne : 0.2738
   Probabilité min : 0.0000
   Probabilité max : 1.0000


In [5]:
# ── Vérification finale vs sample_submission.csv ──────────────────────────────
try:
    sample = pd.read_csv('sample_submission.csv')
    ids_ok  = (submission['id'].values == sample['id'].values).all()
    rows_ok = len(submission) == len(sample)

    print(f"Lignes attendues (sample) : {len(sample):,}")
    print(f"Lignes dans notre fichier : {len(submission):,}")
    print(f"Nombre de lignes correct  : {'OUI' if rows_ok  else 'NON'}")
    print(f"Ordre des IDs correct     : {'OUI' if ids_ok   else 'NON'}")
    print(f"Valeurs dans [0, 1]       : {'OUI' if submission['target'].between(0,1).all() else 'NON'}")

    if rows_ok and ids_ok:
        print("\n Fichier de soumission prêt !")
    else:
        print("\n Vérifier les IDs ou le nombre de lignes !")
except FileNotFoundError:
    print("sample_submission.csv introuvable — vérifie manuellement le nombre de lignes.")
    print(f"Notre fichier : {len(submission):,} lignes")
    print(f"Valeurs dans [0,1] : {submission['target'].between(0,1).all()}")

Lignes attendues (sample) : 430,100
Lignes dans notre fichier : 430,100
Nombre de lignes correct  : OUI
Ordre des IDs correct     : OUI
Valeurs dans [0, 1]       : OUI

 Fichier de soumission prêt !
